# 08 — Análise longitudinal espacial do ENEM

## Objetivo

Investigar a evolução espacial do desempenho municipal no ENEM, por meio de mapas interativos, mapas de variação entre os extremos de cada período e mapas de tendência municipal.

A análise respeita as duas estruturas históricas de avaliação:

| Período | Indicadores | Redação |
|---|---|---|
| **1998–2008** | OBJETIVA | critério antigo |
| **2009–2025** | CN, CH, LC e MT | critério novo |

**Regra metodológica:** não será construída uma série longitudinal contínua entre os períodos. A Redação antiga e a nova são analisadas separadamente; OBJETIVA não é comparada às quatro áreas.

## Estratégia cartográfica

Não será produzido um mapa para cada ano. Serão usados três anos de referência por período, desde que existam dados:

- 1998–2008: 1998, 2003 e 2008.
- 2009–2025: 2009, 2017 e 2025.

Os mapas de referência permitem alternar os anos por meio do controle de camadas. Os mapas de variação mostram a diferença entre o primeiro e o último ano de cada período. Os mapas de tendência representam a inclinação temporal municipal estimada com todos os anos disponíveis.

**Interatividade:** zoom, arraste, identificação do município ao passar o mouse e consulta dos indicadores ao posicionar o cursor sobre cada município.

As análises estatísticas, rankings e distribuições permanecem nos notebooks longitudinais já existentes. Este notebook concentra-se na representação espacial.


## 1. Configuração e carga dos dados

Ajuste a UF no início. A função `read_allfiles()` já realiza a leitura das bases e a harmonização dos nomes municipais com a malha geográfica. Não repetimos essa normalização neste notebook.


**Dependências adicionais:** `folium`, `branca`, `python-dotenv`. A chave `CARTO_BASEMAPS_API_KEY` deve estar em um arquivo `.env` na raiz do projeto, sem ser enviada ao Git.

Instalação, se necessário: `%pip install folium branca python-dotenv`.


In [1]:
# ============================================================
# 1. CONFIGURAÇÃO, IMPORTAÇÕES E LEITURA
# ============================================================

from _setup import *

import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import html as html_lib
from IPython.display import display, HTML

import os
from dotenv import load_dotenv

from branca.colormap import LinearColormap
from branca.element import Figure
from IPython.display import display

from enem_pipeline.config import PROJECT_ROOT, UF_PROCESSAMENTO
from utils.features_names import read_allfiles
from utils.eda import (
    PERIODOS,
    DISCIPLINAS_ANTIGO,
    DISCIPLINAS_NOVO,
    calcular_tendencias_municipais,
)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

# ------------------------------------------------------------
# Parâmetros definidos para esta execução
# ------------------------------------------------------------

# Chave CARTO armazenada localmente no arquivo .env (não versionar).
load_dotenv()
CARTO_BASEMAPS_API_KEY = os.getenv("CARTO_BASEMAPS_API_KEY")
if not CARTO_BASEMAPS_API_KEY:
    raise EnvironmentError("CARTO_BASEMAPS_API_KEY não encontrada no .env.")

CARTO_TILES = (
    "https://{s}.basemaps.cartocdn.com/"
    "light_all/{z}/{x}/{y}.png"
    f"?key={CARTO_BASEMAPS_API_KEY}"
)

CARTO_ATTRIBUTION = (
    '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> '
    'contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'
)

UF_PROCESSAMENTO = "PA"

ANOS_REFERENCIA = {
    "1998_2008": [1998, 2003, 2008],
    "2009_2025": [2009, 2017, 2025],
}

MIN_ANOS_TENDENCIA = 5
ALPHA_TENDENCIA = 0.05

# ------------------------------------------------------------
# Leitura única da camada de Consumo
# ------------------------------------------------------------

out = read_allfiles(UF_PROCESSAMENTO)

painel = out["painel"]["data"].copy()
municipios_geo = out["shape"]["data"].copy()

# ------------------------------------------------------------
# Estruturas históricas e validações iniciais
# ------------------------------------------------------------

DISCIPLINAS_PERIODOS = {
    "1998_2008": DISCIPLINAS_ANTIGO,
    "2009_2025": DISCIPLINAS_NOVO,
}

COLUNA_CODIGO_PAINEL = "CO_MUNICIPIO_TERRITORIO"
COLUNA_CODIGO_SHAPE = "CD_MUN"
COLUNA_NOME_MUNICIPIO = "NO_MUNICIPIO_TERRITORIO"

colunas_obrigatorias = [
    "ano",
    COLUNA_CODIGO_PAINEL,
    COLUNA_NOME_MUNICIPIO,
]

faltantes = [c for c in colunas_obrigatorias if c not in painel.columns]
if faltantes:
    raise KeyError(f"Colunas ausentes no painel municipal: {faltantes}")

if COLUNA_CODIGO_SHAPE not in municipios_geo.columns:
    raise KeyError(
        f"Coluna {COLUNA_CODIGO_SHAPE!r} ausente na malha geográfica."
    )

# Conversão numérica do ano.
painel["ano"] = pd.to_numeric(painel["ano"], errors="coerce")
painel = painel[painel["ano"].notna()].copy()
painel["ano"] = painel["ano"].astype(int)

# Chave auxiliar para junções: não altera os nomes municipais.
def codigo_como_texto(serie):
    return (
        serie.astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

painel["_codigo_municipio_join"] = codigo_como_texto(
    painel[COLUNA_CODIGO_PAINEL]
)
municipios_geo["_codigo_municipio_join"] = codigo_como_texto(
    municipios_geo[COLUNA_CODIGO_SHAPE]
)

# A malha deve possuir um único registro por código municipal.
if municipios_geo["_codigo_municipio_join"].duplicated().any():
    duplicados = municipios_geo.loc[
        municipios_geo["_codigo_municipio_join"].duplicated(keep=False),
        [COLUNA_CODIGO_SHAPE, "_codigo_municipio_join"],
    ]
    raise ValueError(
        "A malha contém códigos municipais duplicados. "
        f"Exemplos:\n{duplicados.head()}"
    )

# O painel deve possuir um único registro por município/ano.
duplicados_painel = painel.duplicated(
    [COLUNA_CODIGO_PAINEL, "ano"],
    keep=False,
)
if duplicados_painel.any():
    exemplos = painel.loc[
        duplicados_painel,
        [COLUNA_CODIGO_PAINEL, COLUNA_NOME_MUNICIPIO, "ano"],
    ].head(10)
    raise ValueError(
        "O painel possui mais de um registro para município/ano. "
        "É necessário auditar essa duplicidade antes da análise espacial. "
        f"Exemplos:\n{exemplos}"
    )

if municipios_geo.crs is None:
    raise ValueError(
        "A malha geográfica não possui CRS definido. "
        "Defina o sistema de referência antes de usar Folium."
    )

anos_disponiveis = sorted(painel["ano"].unique())

print("=" * 80)
print("ANÁLISE LONGITUDINAL ESPACIAL — CAMADA DE CONSUMO")
print("=" * 80)
print(f"Projeto: {PROJECT_ROOT}")
print(f"UF: {UF_PROCESSAMENTO}")
print(f"Painel: {painel.shape[0]:,} linhas × {painel.shape[1]} colunas")
print(f"Municípios na malha: {len(municipios_geo):,}")
print(f"Anos disponíveis no painel: {min(anos_disponiveis)}–{max(anos_disponiveis)}")
print(f"Total de anos disponíveis: {len(anos_disponiveis)}")
print(f"CRS da malha: {municipios_geo.crs}")


ANÁLISE LONGITUDINAL ESPACIAL — CAMADA DE CONSUMO
Projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF: PA
Painel: 2,412 linhas × 83 colunas
Municípios na malha: 144
Anos disponíveis no painel: 1998–2025
Total de anos disponíveis: 28
CRS da malha: EPSG:4674


## 2. Auditoria da cobertura temporal e municipal

Antes de mapear, verificamos quantos municípios possuem média disponível para cada indicador em cada ano. Essa etapa permite identificar lacunas e evita interpretar ausência de dados como desempenho baixo.

A cobertura é calculada a partir dos registros efetivamente existentes no painel municipal.


In [2]:
# ============================================================
# 2. AUDITORIA DA COBERTURA TEMPORAL
# ============================================================

linhas_cobertura = []

for periodo_id, disciplinas in DISCIPLINAS_PERIODOS.items():
    periodo = PERIODOS[periodo_id]

    dados_periodo = painel[
        painel["ano"].between(periodo["inicio"], periodo["fim"])
    ]

    for ano, dados_ano in dados_periodo.groupby("ano"):
        for coluna, info in disciplinas.items():
            if coluna not in dados_ano.columns:
                n_com_dado = 0
            else:
                valores = pd.to_numeric(
                    dados_ano[coluna], errors="coerce"
                )
                n_com_dado = int(valores.notna().sum())

            linhas_cobertura.append({
                "periodo": periodo["rotulo"],
                "periodo_id": periodo_id,
                "ano": int(ano),
                "disciplina": info["nome"],
                "coluna": coluna,
                "n_municipios_com_media": n_com_dado,
                "n_municipios_no_painel": int(
                    dados_ano[COLUNA_CODIGO_PAINEL].nunique()
                ),
            })

cobertura_temporal = pd.DataFrame(linhas_cobertura)

if cobertura_temporal.empty:
    raise ValueError("Não foram encontrados registros nos períodos analíticos.")

display(
    cobertura_temporal.sort_values(
        ["periodo_id", "disciplina", "ano"]
    ).reset_index(drop=True)
)

# Anos de referência efetivamente presentes no painel.
ANOS_REFERENCIA_DISPONIVEIS = {}

for periodo_id, anos_referencia in ANOS_REFERENCIA.items():
    periodo = PERIODOS[periodo_id]
    anos_periodo = set(
        painel.loc[
            painel["ano"].between(periodo["inicio"], periodo["fim"]),
            "ano",
        ].unique()
    )

    anos_validos = [
        ano for ano in anos_referencia if ano in anos_periodo
    ]
    anos_ausentes = [
        ano for ano in anos_referencia if ano not in anos_periodo
    ]

    ANOS_REFERENCIA_DISPONIVEIS[periodo_id] = anos_validos

    print(f"\n{periodo['rotulo']}")
    print(f"Anos de referência disponíveis: {anos_validos}")
    if anos_ausentes:
        print(f"Anos de referência sem registros: {anos_ausentes}")



,periodo,periodo_id,ano,disciplina,coluna,n_municipios_com_media,n_municipios_no_painel
0,1998–2008,1998_2008,1998,OBJETIVA,media_objetiva,5,5
1,1998–2008,1998_2008,1999,OBJETIVA,media_objetiva,60,60
2,1998–2008,1998_2008,2000,OBJETIVA,media_objetiva,78,78
3,1998–2008,1998_2008,2001,OBJETIVA,media_objetiva,116,116
4,1998–2008,1998_2008,2002,OBJETIVA,media_objetiva,128,128
...,...,...,...,...,...,...,...
102,2009–2025,2009_2025,2021,REDACAO,media_redacao,77,77
103,2009–2025,2009_2025,2022,REDACAO,media_redacao,77,77
104,2009–2025,2009_2025,2023,REDACAO,media_redacao,79,79
105,2009–2025,2009_2025,2024,REDACAO,media_redacao,81,81



1998–2008
Anos de referência disponíveis: [1998, 2003, 2008]

2009–2025
Anos de referência disponíveis: [2009, 2017, 2025]


### 2.1 AUDITORIA DE COBERTURA NOS EXTREMOS DOS PERÍODOS

In [3]:
# ============================================================
# AUDITORIA DE COBERTURA NOS EXTREMOS DOS PERÍODOS
# ============================================================

for periodo_id, disciplinas in DISCIPLINAS_PERIODOS.items():

    periodo = PERIODOS[periodo_id]
    ano_inicial = periodo["inicio"]
    ano_final = periodo["fim"]

    print("\n" + "=" * 80)
    print(f"COBERTURA DOS EXTREMOS — {periodo['rotulo']}")
    print("=" * 80)

    for coluna, info in disciplinas.items():

        cobertura = {}

        for ano in [ano_inicial, ano_final]:

            dados = painel[painel["ano"] == ano]

            if coluna not in dados.columns:
                municipios = set()
            else:
                valores = pd.to_numeric(
                    dados[coluna], errors="coerce"
                )

                municipios = set(
                    dados.loc[
                        valores.notna(),
                        COLUNA_CODIGO_PAINEL
                    ].astype(str)
                )

            cobertura[ano] = municipios

        comuns = cobertura[ano_inicial] & cobertura[ano_final]

        print(f"\n{info['nome']}")
        print(f"Municípios em {ano_inicial}: {len(cobertura[ano_inicial])}")
        print(f"Municípios em {ano_final}: {len(cobertura[ano_final])}")
        print(f"Municípios com dados nos dois anos: {len(comuns)}")


COBERTURA DOS EXTREMOS — 1998–2008

OBJETIVA
Municípios em 1998: 5
Municípios em 2008: 66
Municípios com dados nos dois anos: 5

REDACAO
Municípios em 1998: 5
Municípios em 2008: 66
Municípios com dados nos dois anos: 5

COBERTURA DOS EXTREMOS — 2009–2025

REDACAO
Municípios em 2009: 76
Municípios em 2025: 89
Municípios com dados nos dois anos: 69

CN
Municípios em 2009: 76
Municípios em 2025: 89
Municípios com dados nos dois anos: 69

CH
Municípios em 2009: 76
Municípios em 2025: 89
Municípios com dados nos dois anos: 69

LC
Municípios em 2009: 76
Municípios em 2025: 89
Municípios com dados nos dois anos: 69

MT
Municípios em 2009: 76
Municípios em 2025: 89
Municípios com dados nos dois anos: 69


## 3. Funções cartográficas interativas

As funções abaixo padronizam as junções por código municipal, a conversão da malha para WGS84 e a apresentação dos mapas no Folium. Municípios sem valor permanecem em cinza.


In [4]:
# ============================================================
# 3. FUNÇÕES AUXILIARES PARA MAPAS INTERATIVOS (FOLIUM)
# ============================================================

# Paleta de desempenho: médias maiores em azul.
# CORES_DESEMPENHO = [
#     "#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"
# ]

CORES_DESEMPENHO = [
   "#d7191c","#fdae61", "#ffffbf", "#abd9e9" ,"#2c7bb6"
]


# Paleta divergente RdBu: valores negativos em vermelho,
# zero em branco e valores positivos em azul.
CORES_TENDENCIA_RDBU = [
    "#b2182b", "#ef8a62", "#f7f7f7", "#67a9cf", "#2166ac"
]


def preparar_mapa_municipal(dados, coluna_valor, nome_coluna_saida=None):
    """Junta variável municipal à malha usando o código territorial."""

    nome_coluna_saida = nome_coluna_saida or coluna_valor

    colunas_base = [COLUNA_CODIGO_PAINEL, coluna_valor]
    if COLUNA_NOME_MUNICIPIO in dados.columns:
        colunas_base.append(COLUNA_NOME_MUNICIPIO)
    base = dados[colunas_base].copy()

    base["_codigo_municipio_join"] = codigo_como_texto(
        base[COLUNA_CODIGO_PAINEL]
    )
    base = base.drop(columns=[COLUNA_CODIGO_PAINEL])
    base = base.rename(columns={coluna_valor: nome_coluna_saida})

    mapa = municipios_geo.merge(
        base,
        on="_codigo_municipio_join",
        how="left",
        validate="one_to_one",
    )

    return mapa
    
def adicionar_legenda_html(mapa, colormap, titulo, n_cores=24, casas=1):
    """Legenda compacta em gradiente, com mín / meio / máx visíveis."""

    vmin, vmax = float(colormap.vmin), float(colormap.vmax)
    passos = [vmin + (vmax - vmin) * i / (n_cores - 1) for i in range(n_cores)]
    gradiente = ", ".join(colormap(v) for v in passos)
    meio = (vmin + vmax) / 2

    html = f"""
    <div style="position: absolute; top: 10px; right: 10px; z-index: 9999;
                width: 190px; max-width: 60%; box-sizing: border-box;
                background: rgba(255,255,255,0.1); padding: 6px 8px;
                border-radius: 4px; box-shadow: 0 0 4px rgba(0,0,0,0.1);
                font-family: Arial; font-size: 11px; color: #333;">
        <div style="margin-bottom: 4px; font-weight: bold;">{titulo}</div>
        <div style="height: 12px; border: 1px solid #777;
                    background: linear-gradient(to right, {gradiente});"></div>
        <div style="display: flex; justify-content: space-between;
                    margin-top: 2px;">
            <span>{vmin:.{casas}f}</span>
            <span>{meio:.{casas}f}</span>
            <span>{vmax:.{casas}f}</span>
        </div>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(html))

def criar_mapa_folium(mapa_geo, zoom_start=6):
    """Cria mapa centrado na extensão da malha municipal."""

    mapa_wgs84 = mapa_geo.to_crs(epsg=4326)
    minx, miny, maxx, maxy = mapa_wgs84.total_bounds

    centro = [(minx + maxx) / 2, (miny + maxy) / 2]

    mapa = folium.Map(
        location=centro,
        zoom_start=zoom_start,
        tiles=CARTO_TILES,
        attr=CARTO_ATTRIBUTION,
        subdomains="abcd",
        control_scale=True,
        prefer_canvas=True,
        width="100%",
        height="500px",
    )

    mapa.fit_bounds([[miny, minx], [maxy, maxx]])
    return mapa, mapa_wgs84


def criar_colormap(valores, tipo="desempenho"):
    """Cria escala contínua para desempenho ou variação/tendência."""

    valores = pd.to_numeric(
        pd.Series(valores), errors="coerce"
    ).dropna()

    if valores.empty:
        vmin, vmax = 0.0, 1.0
    else:
        vmin = float(valores.min())
        vmax = float(valores.max())

    if tipo == "divergente":
        limite = max(abs(vmin), abs(vmax))
        if not np.isfinite(limite) or limite == 0:
            limite = 1.0

        return LinearColormap(
            colors=CORES_DESEMPENHO,
            index=[
                -limite,
                -limite / 2,
                0,
                limite / 2,
                limite,
            ],
            vmin=-limite,
            vmax=limite,
        )

    if vmax == vmin:
        vmin -= 0.5
        vmax += 0.5

    return LinearColormap(
        colors=CORES_DESEMPENHO,
        vmin=vmin,
        vmax=vmax,
    )



def adicionar_camada_municipal(
    mapa_folium,
    mapa_geo,
    coluna_valor,
    nome_camada,
    colormap,
    campos_tooltip,
    aliases_tooltip,
    mostrar=True,
):
    """Adiciona GeoJSON municipal com preenchimento, tooltip e destaque."""

    dados_geo = mapa_geo.copy()

    # Folium/GeoJSON trabalha em WGS84.
    if dados_geo.crs is None or dados_geo.crs.to_epsg() != 4326:
        dados_geo = dados_geo.to_crs(epsg=4326)

    # Evita NaN/inf no GeoJSON.
    dados_geo[coluna_valor] = pd.to_numeric(
        dados_geo[coluna_valor], errors="coerce"
    )
    dados_geo[coluna_valor] = dados_geo[coluna_valor].replace(
        [np.inf, -np.inf], np.nan
    )

    campos_validos = [
        campo for campo in campos_tooltip
        if campo in dados_geo.columns
    ]
    aliases_validos = [
        alias for campo, alias in zip(campos_tooltip, aliases_tooltip)
        if campo in dados_geo.columns
    ]

    def estilo(feature):
        valor = feature["properties"].get(coluna_valor)

        if valor is None:
            cor = "#d9d9d9"
            preenchimento = 0.35
        else:
            try:
                cor = colormap(float(valor))
                preenchimento = 0.78
            except (TypeError, ValueError):
                cor = "#d9d9d9"
                preenchimento = 0.35

        return {
            "fillColor": cor,
            "color": "#555555",
            "weight": 0.7,
            "fillOpacity": preenchimento,
        }

    camada = folium.FeatureGroup(
        name=nome_camada,
        show=mostrar,
    )

    folium.GeoJson(
        data=dados_geo.to_json(),
        name=nome_camada,
        style_function=estilo,
        highlight_function=lambda feature: {
            "weight": 2.2,
            "color": "#222222",
            "fillOpacity": 0.95,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=campos_validos,
            aliases=aliases_validos,
            localize=True,
            sticky=False,
            labels=True,
            style=(
                "background-color: white; color: #333333; "
                "font-family: Arial; font-size: 12px; padding: 8px;"
            ),
        ) if campos_validos else None,
    ).add_to(camada)

    camada.add_to(mapa_folium)
    return camada


def finalizar_mapa_folium(mapa, colormap, largura="100%"):
    """Adiciona legenda e controle de camadas ao mapa."""

    colormap.add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)
    display(mapa)


def formatar_numero(valor, casas=2):
    if pd.isna(valor):
        return "Sem dado"
    return f"{float(valor):.{casas}f}"


def gerar_mapas_por_ano(painel,coluna,nome_disciplina,anos_referencia=None,n_mapas=3):
    """
    Gera `n_mapas` mapas folium (padrão 3) lado a lado para `coluna`.

    O período é inferido automaticamente a partir de `anos_referencia`
    (min–max). Se `anos_referencia` for None, escolhe `n_mapas` anos
    igualmente espaçados entre os anos com dados disponíveis para
    `coluna` — o que, na prática, identifica sozinho o período
    (enem antigo 1998–2008 / enem novo 2009–2025).
    """

    if coluna not in painel.columns:
        raise KeyError(f"Coluna {coluna!r} não encontrada no painel.")

    anos_disponiveis = set(
        pd.to_numeric(painel['ano'], errors='coerce')
          .dropna().astype(int)
    )

    # ---------- 1. Resolve anos de referência ----------
    if anos_referencia is None:
        anos_com_dados = sorted(
            pd.to_numeric(
                painel.loc[painel[coluna].notna(), 'ano'],
                errors='coerce',
            ).dropna().astype(int).unique()
        )
        if not anos_com_dados:
            print(f'{nome_disciplina}: sem anos com dados disponíveis.')
            return None

        if len(anos_com_dados) <= n_mapas:
            anos_referencia = tuple(anos_com_dados)
        else:
            passo = (len(anos_com_dados) - 1) / (n_mapas - 1)
            idx = [round(i * passo) for i in range(n_mapas)]
            anos_referencia = tuple(anos_com_dados[i] for i in idx)

    anos_referencia = tuple(int(a) for a in anos_referencia)

    if len(anos_referencia) < 2:
        raise ValueError('anos_referencia deve conter pelo menos 2 anos.')

    # ---------- 2. Período inferido ----------
    periodo = f"{min(anos_referencia)}–{max(anos_referencia)}"

    # ---------- 3. Filtra anos efetivamente presentes ----------
    anos_mapa = [a for a in anos_referencia if a in anos_disponiveis]
    if not anos_mapa:
        print('Nenhum ano de referência disponível.')
        return None

    # ---------- 4. Valores para o colormap ----------
    valores_periodo = pd.to_numeric(
        painel.loc[painel['ano'].isin(anos_mapa), coluna],
        errors='coerce',
    ).dropna()

    if valores_periodo.empty:
        print(f'{nome_disciplina}: sem valores nos anos de referência.')
        return None

    # ---------- 5. Construção dos mapas ----------
    mapas = []
    for ano in anos_mapa:
        colormap_ano = criar_colormap(valores_periodo, tipo='desempenho')
        #colormap_ano.caption = (
        #    f'Média municipal de {nome_disciplina} — {periodo}'
        #)
        titulo_legenda = f'Média de {nome_disciplina} ({periodo})'

        dados_ano = painel.loc[painel['ano'] == ano].copy()
        dados_ano[coluna] = pd.to_numeric(dados_ano[coluna], errors='coerce')

        mapa_geo = preparar_mapa_municipal(
            dados_ano, coluna, nome_coluna_saida='media_municipal',
        )
        mapa_geo['ano_mapa'] = int(ano)
        mapa_geo['media_texto'] = mapa_geo['media_municipal'].map(
            lambda x: formatar_numero(x, 2)
        )

        mapa_folium, _ = criar_mapa_folium(municipios_geo)
        adicionar_camada_municipal(
            mapa_folium=mapa_folium,
            mapa_geo=mapa_geo,
            coluna_valor='media_municipal',
            nome_camada=str(ano),
            colormap=colormap_ano,
            campos_tooltip=[COLUNA_NOME_MUNICIPIO, 'ano_mapa', 'media_texto'],
            aliases_tooltip=['Município:', 'Ano:', f'Média de {nome_disciplina}:'],
        )
        
        #colormap_ano.add_to(mapa_folium)
        adicionar_legenda_html(mapa_folium, colormap_ano, titulo_legenda)

        mapas.append((ano, mapa_folium))

    # ---------- 6. Montagem do HTML ----------
    blocos_html = []
    for ano, mapa_folium in mapas:
        conteudo_escapado = html_lib.escape(
            mapa_folium.get_root().render(), quote=True,
        )
        blocos_html.append(f"""
            <div style="flex: 1 1 0; min-width: 0; box-sizing: border-box;">
                <h4 style="text-align: center; margin: 4px 0 8px 0;">{ano}</h4>
                <iframe srcdoc="{conteudo_escapado}"
                        style="display: block; width: 100%; height: 500px;
                               border: none;"
                        loading="eager"></iframe>
            </div>
        """)

    html_final = f"""
        <div style="display: flex; flex-direction: row; gap: 8px;
                    width: 100%; align-items: flex-start;
                    box-sizing: border-box;">
            {''.join(blocos_html)}
        </div>
    """
    display(HTML(html_final))
    print(f'{nome_disciplina}: {len(mapas)} mapas exibidos '
          f'(período {periodo}).')

    return mapas


## 4. Mapas interativos dos anos de referência

Cada disciplina é apresentada em uma célula independente. Em cada célula, os três anos de referência aparecem em mapas Folium lado a lado, permitindo comparação visual direta.

A escala de cores é comum aos três anos de cada disciplina. Valores menores aparecem em azul e valores maiores em vermelho. Cada mapa pode ser ampliado e navegado independentemente.


### 4.1 1998–2008 — OBJETIVA 

In [1]:
# ============================================================
# MAPAS DE REFERÊNCIA — 1998–2008 — OBJETIVA 
# ============================================================
anos_referencia=(1998, 2001,2004,2008)
mapas = gerar_mapas_por_ano(painel, 'media_objetiva', 'Objetiva',anos_referencia)

### 4.2 1998–2008 — Redação 

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 1998–2008 — Redação
# ============================================================

anos_referencia=(1998, 2001,2004,2008)
mapas = gerar_mapas_por_ano(painel, 'media_redacao', 'Redacao',anos_referencia)

### 4.3 2009–2025 — Redação

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 2009–2025 — Redação
# ============================================================
anos_referencia=(2009, 2014,2019,2025)
mapas = gerar_mapas_por_ano(painel, 'media_redacao', 'Redacao',anos_referencia)


### 4.4 2009–2025 — CN

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 2009–2025 — CN
# ============================================================

coluna = 'media_cn'
nome_disciplina = 'CN'
anos_referencia=(2009, 2014,2019,2025)
mapas = gerar_mapas_por_ano(painel, coluna, nome_disciplina,anos_referencia)

### 4.5 2009–2025 — CH

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 2009–2025 — CH
# ============================================================

coluna = 'media_ch'
nome_disciplina = 'CH'

anos_referencia=(2009, 2014,2019,2025)
mapas = gerar_mapas_por_ano(painel, coluna, nome_disciplina,anos_referencia)

### 4.6 2009–2025 — LC

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 2009–2025 — LC
# ============================================================
coluna = 'media_lc'
nome_disciplina = 'LC'
anos_referencia=(2009, 2014,2019, 2025)

mapas = gerar_mapas_por_ano(painel,coluna,nome_disciplina,anos_referencia)

### 4.7 2009–2025 — MT

In [ ]:
# ============================================================
# MAPAS DE REFERÊNCIA — 2009–2025 — MT
# ============================================================

coluna = 'media_mt'
nome_disciplina = 'MT'

anos_referencia=(2009, 2014,2019,2025)
mapas = gerar_mapas_por_ano(painel, coluna, nome_disciplina,anos_referencia)

## 5. Mapas interativos de variação entre os extremos de cada período

A variação é calculada como a média do ano final menos a média do ano inicial. São considerados apenas municípios com valores disponíveis nos dois anos.

A escala divergente é centrada em zero: vermelho indica redução da média, branco indica variação próxima de zero e azul indica crescimento da média.


In [ ]:
# ============================================================
# 5. MAPAS INTERATIVOS DE VARIAÇÃO ENTRE OS EXTREMOS
# ============================================================

def exibir_grade_mapas_folium(mapas, titulo_periodo, altura=500):
    """
    Finaliza e exibe mapas Folium em uma grade com duas colunas.

    A finalização mantém o padrão do Bloco 4:
    - legenda (colormap);
    - controle de camadas;
    - mapa interativo com tooltip.

    Cada item de `mapas` deve ser:
        (titulo, mapa_folium, colormap)
    """

    if not mapas:
        print(f"Nenhum mapa disponível para {titulo_periodo}.")
        return

    blocos_html = []

    for titulo, mapa_folium, colormap in mapas:

        # Mesmas etapas de finalização adotadas no Bloco 4.
        colormap.add_to(mapa_folium)
        #folium.LayerControl(collapsed=False).add_to(mapa_folium)

        conteudo_escapado = html_lib.escape(
            mapa_folium.get_root().render(),
            quote=True,
        )

        blocos_html.append(f"""
            <div style="
                min-width: 0;
                width: 100%;
                box-sizing: border-box;
            ">
                <h4 style="
                    text-align: center;
                    margin: 4px 0 8px 0;
                ">{titulo}</h4>

                <iframe
                    srcdoc="{conteudo_escapado}"
                    style="
                        display: block;
                        width: 100%;
                        height: {altura}px;
                        border: none;
                    "
                    loading="eager">
                </iframe>
            </div>
        """)

    # Mantém duas colunas. Se houver quantidade ímpar de mapas,
    # a última posição da grade permanece vazia.
    if len(blocos_html) % 2 != 0:
        blocos_html.append("""
            <div style="
                min-width: 0;
                width: 100%;
                min-height: 1px;
                box-sizing: border-box;
            "></div>
        """)

    html_final = f"""
        <h3 style="margin: 12px 0;">{titulo_periodo}</h3>
        <div style="
            display: grid;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            gap: 12px;
            width: 100%;
            align-items: start;
            box-sizing: border-box;
        ">
            {''.join(blocos_html)}
        </div>
    """

    display(HTML(html_final))


# Ordem explícita de apresentação, independente da ordem dos dicionários.
ORDEM_DISCIPLINAS_MAPAS = {
    "1998_2008": [
        "media_objetiva",
        "media_redacao",
    ],
    "2009_2025": [
        "media_lc",
        "media_redacao",
        "media_mt",
        "media_cn",
        "media_ch",
    ],
}

for periodo_id, disciplinas in DISCIPLINAS_PERIODOS.items():

    periodo = PERIODOS[periodo_id]
    ano_inicial = periodo["inicio"]
    ano_final = periodo["fim"]

    anos_existentes = set(painel["ano"].unique())

    if ano_inicial not in anos_existentes or ano_final not in anos_existentes:
        print(
            f"Variação {periodo['rotulo']} ignorada: "
            f"não há registros em {ano_inicial} e/ou {ano_final}."
        )
        continue

    print("\n" + "=" * 80)
    print(f"MAPAS INTERATIVOS DE VARIAÇÃO — {ano_inicial} → {ano_final}")
    print("=" * 80)

    mapas_periodo = []

    for coluna in ORDEM_DISCIPLINAS_MAPAS[periodo_id]:

        if coluna not in disciplinas:
            raise KeyError(
                f"A disciplina {coluna!r} não está configurada "
                f"em DISCIPLINAS_PERIODOS[{periodo_id!r}]."
            )

        info = disciplinas[coluna]

        base_inicial = painel.loc[
            painel["ano"] == ano_inicial,
            [COLUNA_CODIGO_PAINEL, coluna],
        ].copy()

        base_final = painel.loc[
            painel["ano"] == ano_final,
            [COLUNA_CODIGO_PAINEL, coluna],
        ].copy()

        base_inicial[coluna] = pd.to_numeric(
            base_inicial[coluna], errors="coerce"
        )
        base_final[coluna] = pd.to_numeric(
            base_final[coluna], errors="coerce"
        )

        base_inicial = base_inicial.rename(
            columns={coluna: "media_inicial"}
        )
        base_final = base_final.rename(
            columns={coluna: "media_final"}
        )

        variacao = base_inicial.merge(
            base_final,
            on=COLUNA_CODIGO_PAINEL,
            how="inner",
            validate="one_to_one",
        )

        variacao = variacao.dropna(
            subset=["media_inicial", "media_final"]
        )

        if variacao.empty:
            print(
                f"{info['nome']}: nenhum município com dados "
                f"nos dois extremos."
            )
            continue

        variacao["delta"] = (
            variacao["media_final"] - variacao["media_inicial"]
        )

        mapa_geo = preparar_mapa_municipal(
            variacao,
            "delta",
            nome_coluna_saida="variacao_pontos",
        )

        # Recupera as médias inicial e final pela chave municipal.
        variacao["_codigo_municipio_join"] = codigo_como_texto(
            variacao[COLUNA_CODIGO_PAINEL]
        )
        variacao_indexada = variacao.set_index(
            "_codigo_municipio_join"
        )

        mapa_geo["media_inicial_texto"] = mapa_geo[
            "_codigo_municipio_join"
        ].map(variacao_indexada["media_inicial"])

        mapa_geo["media_final_texto"] = mapa_geo[
            "_codigo_municipio_join"
        ].map(variacao_indexada["media_final"])

        mapa_geo["variacao_texto"] = mapa_geo[
            "variacao_pontos"
        ].map(lambda x: formatar_numero(x, 2))

        mapa_geo["media_inicial_texto"] = mapa_geo[
            "media_inicial_texto"
        ].map(lambda x: formatar_numero(x, 2))

        mapa_geo["media_final_texto"] = mapa_geo[
            "media_final_texto"
        ].map(lambda x: formatar_numero(x, 2))

        colormap = criar_colormap(
            mapa_geo["variacao_pontos"],
            tipo="divergente",
        )
        colormap.caption = (
            f"Variação em pontos: {ano_final} − {ano_inicial} "
            "(vermelho = redução; azul = crescimento)"
        )

        mapa_folium, _ = criar_mapa_folium(municipios_geo)

        adicionar_camada_municipal(
            mapa_folium=mapa_folium,
            mapa_geo=mapa_geo,
            coluna_valor="variacao_pontos",
            nome_camada=info["nome"],
            colormap=colormap,
            campos_tooltip=[
                COLUNA_NOME_MUNICIPIO,
                "media_inicial_texto",
                "media_final_texto",
                "variacao_texto",
            ],
            aliases_tooltip=[
                "Município:",
                f"Média {ano_inicial}:",
                f"Média {ano_final}:",
                "Variação (pontos):",
            ],
        )

        mapas_periodo.append(
            (info["nome"], mapa_folium, colormap)
        )

        print(
            f"{info['nome']}: {len(variacao)} municípios com dados "
            f"nos dois extremos."
        )

    exibir_grade_mapas_folium(
        mapas_periodo,
        titulo_periodo=f"Variação municipal — {periodo['rotulo']}",
    )


## 6. Mapas interativos de tendência municipal

Para cada município e indicador, a regressão linear é estimada com os anos disponíveis dentro do período. O mapa representa a inclinação em pontos por ano.

A escala segue a convenção definida para este projeto: **azul = inclinação positiva (crescimento da média); vermelho = inclinação negativa (redução da média)**. Municípios sem tendência com `p < ALPHA_TENDENCIA` permanecem em cinza, preservando a regra de visualização adotada na versão anterior.

Ao passar o cursor sobre um município, o tooltip apresenta nome, inclinação, R², valor-p, número de anos utilizados e intervalo temporal efetivamente utilizado.

A significância nominal deve ser interpretada com cautela: a regressão linear convencional não corrige automaticamente autocorrelação temporal nem múltiplos testes municipais.


In [ ]:
# ============================================================
# 6. MAPAS INTERATIVOS DE TENDÊNCIA LINEAR MUNICIPAL
# ============================================================

for periodo_id, disciplinas in DISCIPLINAS_PERIODOS.items():

    periodo = PERIODOS[periodo_id]

    dados_periodo = painel[
        painel["ano"].between(periodo["inicio"], periodo["fim"])
    ].copy()

    if dados_periodo["ano"].nunique() < MIN_ANOS_TENDENCIA:
        print(
            f"Tendência {periodo['rotulo']} ignorada: "
            f"menos de {MIN_ANOS_TENDENCIA} anos disponíveis."
        )
        continue

    print("\n" + "=" * 80)
    print(f"MAPAS INTERATIVOS DE TENDÊNCIA — {periodo['rotulo']}")
    print("=" * 80)

    tendencias_periodo = calcular_tendencias_municipais(
        painel,
        periodo,
        disciplinas,
        min_anos=MIN_ANOS_TENDENCIA,
    )

    if tendencias_periodo.empty:
        print(f"Sem tendências municipais calculáveis em {periodo['rotulo']}.")
        continue

    mapas_periodo = []

    for coluna in ORDEM_DISCIPLINAS_MAPAS[periodo_id]:

        if coluna not in disciplinas:
            raise KeyError(
                f"A disciplina {coluna!r} não está configurada "
                f"em DISCIPLINAS_PERIODOS[{periodo_id!r}]."
            )

        info = disciplinas[coluna]

        dados_tendencia = tendencias_periodo[
            tendencias_periodo["disciplina"] == info["nome"]
        ].copy()

        if dados_tendencia.empty:
            print(f"{info['nome']}: sem tendências municipais calculáveis.")
            continue

        dados_tendencia["slope_pontos_ano"] = pd.to_numeric(
            dados_tendencia["slope_pontos_ano"], errors="coerce"
        )
        dados_tendencia["p_valor"] = pd.to_numeric(
            dados_tendencia["p_valor"], errors="coerce"
        )

        # Mantém a regra já adotada: somente p < alpha recebe cor.
        dados_tendencia["slope_mapa"] = dados_tendencia[
            "slope_pontos_ano"
        ].where(
            dados_tendencia["p_valor"] < ALPHA_TENDENCIA
        )

        mapa_geo = preparar_mapa_municipal(
            dados_tendencia,
            "slope_mapa",
            nome_coluna_saida="slope_mapa",
        )

        # Anexa os indicadores estatísticos ao mapa pelo código municipal.
        detalhes = dados_tendencia.copy()
        detalhes["_codigo_municipio_join"] = codigo_como_texto(
            detalhes["CO_MUNICIPIO_TERRITORIO"]
        )
        detalhes = detalhes.drop_duplicates(
            "_codigo_municipio_join"
        ).set_index("_codigo_municipio_join")

        colunas_detalhe = [
            "slope_pontos_ano",
            "r2",
            "p_valor",
            "n_anos",
            "ano_inicial",
            "ano_final",
        ]

        for coluna_detalhe in colunas_detalhe:
            mapa_geo[coluna_detalhe] = mapa_geo[
                "_codigo_municipio_join"
            ].map(detalhes[coluna_detalhe])

        mapa_geo["inclinação_texto"] = mapa_geo[
            "slope_pontos_ano"
        ].map(lambda x: formatar_numero(x, 3))

        mapa_geo["r2_texto"] = mapa_geo["r2"].map(
            lambda x: formatar_numero(x, 3)
        )

        mapa_geo["p_valor_texto"] = mapa_geo["p_valor"].map(
            lambda x: formatar_numero(x, 5)
        )

        mapa_geo["n_anos_texto"] = mapa_geo["n_anos"].map(
            lambda x: "Sem estimativa" if pd.isna(x) else str(int(x))
        )

        mapa_geo["periodo_utilizado"] = mapa_geo.apply(
            lambda linha: (
                "Sem estimativa"
                if pd.isna(linha["ano_inicial"]) or pd.isna(linha["ano_final"])
                else f"{int(linha['ano_inicial'])}–{int(linha['ano_final'])}"
            ),
            axis=1,
        )

        colormap = criar_colormap(
            mapa_geo["slope_mapa"],
            tipo="divergente",
        )
        colormap.caption = (
            "Inclinação (pontos/ano) — vermelho = redução; "
            "branco = próximo de zero; azul = crescimento"
        )

        mapa_folium, _ = criar_mapa_folium(municipios_geo)

        adicionar_camada_municipal(
            mapa_folium=mapa_folium,
            mapa_geo=mapa_geo,
            coluna_valor="slope_mapa",
            nome_camada=f"Tendência — {info['nome']}",
            colormap=colormap,
            campos_tooltip=[
                COLUNA_NOME_MUNICIPIO,
                "inclinação_texto",
                "r2_texto",
                "p_valor_texto",
                "n_anos_texto",
                "periodo_utilizado",
            ],
            aliases_tooltip=[
                "Município:",
                "Inclinação (pontos/ano):",
                "R²:",
                "Valor-p:",
                "Anos utilizados:",
                "Intervalo utilizado:",
            ],
        )

        n_municipios = dados_tendencia[
            "CO_MUNICIPIO_TERRITORIO"
        ].nunique()

        n_significativos = int(
            (
                dados_tendencia["p_valor"] < ALPHA_TENDENCIA
            ).sum()
        )

        print(
            f"{info['nome']}: {n_municipios} municípios com tendência estimada; "
            f"{n_significativos} com p < {ALPHA_TENDENCIA}."
        )

        mapas_periodo.append(
            (info["nome"], mapa_folium, colormap)
        )

    exibir_grade_mapas_folium(
        mapas_periodo,
        titulo_periodo=f"Tendência municipal — {periodo['rotulo']}",
    )


## 7. Síntese metodológica e limitações

1. **Períodos independentes:** 1998–2008 e 2009–2025 não são conectados em uma única série de desempenho.
2. **Redação:** os critérios antigo e novo são tratados separadamente.
3. **Anos de referência:** são marcos de visualização, não uma amostragem estatística de todos os anos.
4. **Variação:** mapas de diferença usam apenas municípios com valores válidos nos dois extremos.
5. **Tendência:** a inclinação linear resume a evolução observada, mas não demonstra causalidade.
6. **Significância:** os mapas de tendência destacam `p < 0,05`, sem correção para múltiplos testes. Devem ser interpretados como exploração espacial.
7. **Cobertura:** municípios sem cor podem representar ausência de dados, cobertura insuficiente ou tendência não significativa, conforme a seção.
8. **Referência territorial:** a malha municipal é a carregada por `read_allfiles()`. A compatibilidade histórica dos códigos territoriais deve ser considerada na interpretação.

Este notebook é exclusivamente cartográfico. As tabelas analíticas, estatísticas descritivas, rankings e interpretações longitudinais permanecem no notebook de análise longitudinal periodizada.


In [ ]:
# ============================================================
# AUDITORIA DOS RESULTADOS DE TENDÊNCIA
# ============================================================

for periodo_id, disciplinas in DISCIPLINAS_PERIODOS.items():

    periodo = PERIODOS[periodo_id]

    tendencias = calcular_tendencias_municipais(
        painel,
        periodo,
        disciplinas,
        min_anos=MIN_ANOS_TENDENCIA,
    )

    print("\n" + "=" * 90)
    print(f"AUDITORIA DE TENDÊNCIAS — {periodo['rotulo']}")
    print("=" * 90)

    for disciplina in tendencias["disciplina"].unique():

        dados = tendencias[
            tendencias["disciplina"] == disciplina
        ].copy()

        significativos = dados[
            dados["p_valor"] < ALPHA_TENDENCIA
        ]

        print(f"\nIndicador: {disciplina}")
        print(f"Municípios estimados: {len(dados)}")
        print(f"Tendências significativas: {len(significativos)}")

        print(
            f"Tendências positivas: "
            f"{(dados['slope_pontos_ano'] > 0).sum()}"
        )

        print(
            f"Tendências negativas: "
            f"{(dados['slope_pontos_ano'] < 0).sum()}"
        )

        print(
            f"Quantidade de anos utilizados — mínimo: "
            f"{dados['n_anos'].min()}, "
            f"mediana: {dados['n_anos'].median()}, "
            f"máximo: {dados['n_anos'].max()}"
        )

        print(
            f"Ano inicial utilizado — mínimo: "
            f"{dados['ano_inicial'].min()}, "
            f"máximo: {dados['ano_inicial'].max()}"
        )

        print(
            f"Ano final utilizado — mínimo: "
            f"{dados['ano_final'].min()}, "
            f"máximo: {dados['ano_final'].max()}"
        )

        display(
            dados[
                [
                    "NO_MUNICIPIO_TERRITORIO",
                    "n_anos",
                    "ano_inicial",
                    "ano_final",
                    "slope_pontos_ano",
                    "r2",
                    "p_valor",
                ]
            ].sort_values("p_valor").head(10)
        )